# Can synthetic damage teach a model to spot *real* engine-blade damage?

**The question:** real photos of damaged aero-engine blades are rare. If we
train a defect detector mostly on *fake* (synthetically damaged) blades, how
well does it spot *real* damage — and how many real examples does it still need?

**The rule that keeps this honest:** the model is always tested on **real**
damaged blades it has never seen. The synthetic images are only ever training
material.

**Data**
- `synthetic/` — real undamaged AeBAD blades, half left clean, half with damage drawn on by `synth_damage.py`. This is the *training* pool.
- `AeBAD_S/test/` — **real** blades, both good and damaged (ablation, breakdown, fracture, groove). The damaged ones are split into a small "real training" pool and a locked **test set**.

Run the cells top to bottom. On a free Colab **GPU** runtime the whole thing
takes roughly 30–60 minutes. (Runtime → Change runtime type → T4 GPU.)


## 1. Mount Google Drive and unzip the data

Before running this, upload **`synthetic.zip`** and **`aebad_test.zip`** to a
folder called `rolls-royce/` in your Google Drive (see the chat for the `zip`
commands).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, glob

DRIVE_DIR = '/content/drive/MyDrive/rolls-royce'   # change if you used another folder
WORK = '/content/data'
os.makedirs(WORK, exist_ok=True)

for name in ['synthetic.zip', 'aebad_test.zip']:
    src = os.path.join(DRIVE_DIR, name)
    assert os.path.exists(src), f'Missing {src} — upload it to Drive first.'
    with zipfile.ZipFile(src) as z:
        z.extractall(WORK)
    print('unzipped', name)

print('\nTop-level folders now in', WORK, ':')
for p in sorted(glob.glob(WORK + '/*')):
    print(' ', p)

## 2. Setup

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
import torchvision as tv
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')

# ---- config you can tweak -------------------------------------------------
IMG_SIZE   = 224                 # ResNet input size
N_REAL     = [0, 10, 25, 50, 100]# how many REAL damaged images to add to training
SEEDS      = [0, 1, 2]           # repeats, because small data is noisy
EPOCHS     = 6
QUICK      = False               # True = 1 seed, 3 epochs (fast first pass)
if QUICK:
    SEEDS, EPOCHS = [0], 3
# ---------------------------------------------------------------------------
# auto-find the data wherever it unzipped to, whatever the folders are named
_DEF = ['good', 'ablation', 'breakdown', 'fracture', 'groove']
SYNTH_DIR = next(p.parent for p in Path(WORK).rglob('labels.csv')
                 if (p.parent / 'images').is_dir())
TEST_DIR = max((p for p in Path(WORK).rglob('*') if p.is_dir()
                and sum((p / s).is_dir() for s in _DEF) >= 2),
               key=lambda p: sum((p / s).is_dir() for s in _DEF))
print('Synthetic dir:', SYNTH_DIR)
print('Real test dir:', TEST_DIR)

## 3. Index the images and build the splits

**No leakage:** the locked test set of real damaged blades is chosen first and
never used for training. The "real training" images are drawn only from what's
left over. The clean (undamaged) negatives come from the abundant synthetic
clean pool for training, and from real `test/good` for testing.

In [ ]:
DEFECTS = ['ablation', 'breakdown', 'fracture', 'groove']

# --- synthetic training pool (from labels.csv) ---
synth = pd.read_csv(SYNTH_DIR / 'labels.csv')
synth['path'] = synth['filename'].apply(lambda f: str(SYNTH_DIR / 'images' / f))
synth_damaged = synth[synth.label == 1].reset_index(drop=True)
synth_clean   = synth[synth.label == 0].reset_index(drop=True)
print(f'Synthetic: {len(synth_damaged)} damaged, {len(synth_clean)} clean')

# --- real images from the AeBAD test folder ---
rows = []
for p in TEST_DIR.rglob('*'):
    if p.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp'}:
        continue
    parts = {x.lower() for x in p.parts}
    is_good = 'good' in parts
    dtype = next((d for d in DEFECTS if d in parts), None)
    if is_good:
        rows.append({'path': str(p), 'label': 0, 'defect': 'good'})
    elif dtype:
        rows.append({'path': str(p), 'label': 1, 'defect': dtype})
real = pd.DataFrame(rows)
print('\nReal images found:')
print(real.groupby(['defect']).size())

# --- lock a test set: half of each real defect type + half of real good ---
rng = np.random.default_rng(42)
def half_split(df):
    idx = rng.permutation(len(df))
    cut = len(df) // 2
    return df.iloc[idx[:cut]], df.iloc[idx[cut:]]

real_test_parts, real_pool_parts = [], []
for d, grp in real.groupby('defect'):
    test_part, pool_part = half_split(grp)
    real_test_parts.append(test_part)
    real_pool_parts.append(pool_part)        # leftover -> can be added to training
real_test = pd.concat(real_test_parts).reset_index(drop=True)
real_pool = pd.concat(real_pool_parts).reset_index(drop=True)
real_pool_damaged = real_pool[real_pool.label == 1].reset_index(drop=True)

print(f'\nLOCKED TEST SET: {len(real_test)} images '
      f'({(real_test.label==1).sum()} damaged, {(real_test.label==0).sum()} good)')
print(f'Real damaged available to add to training: {len(real_pool_damaged)}')

## 4. Dataset and image transforms

Every image — synthetic or real — is resized to 224×224 and normalised the way
ResNet expects. Training images get light augmentation (flips, small rotations);
test images do not.

In [ ]:
NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.1, 0.1),
    transforms.ToTensor(), NORM])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), NORM])

class BladeDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        return self.tf(img), int(r['label'])

## 5. Model, training and evaluation

A ResNet-18 pretrained on ImageNet, with its final layer swapped for a 2-class
(good / damaged) head. We fine-tune the whole thing briefly. The headline metric
is **recall on real damage** — the fraction of real defects the model catches —
because in inspection a *missed* crack is the dangerous, expensive error.

In [ ]:
def make_model():
    m = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1)
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m.to(DEVICE)

def train_model(train_df, epochs, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = make_model()
    # class weights: counter the clean/damaged imbalance in the training set
    n_dmg = max(1, (train_df.label == 1).sum()); n_cln = max(1, (train_df.label == 0).sum())
    w = torch.tensor([1.0 / n_cln, 1.0 / n_dmg], dtype=torch.float32, device=DEVICE)
    w = w / w.sum() * 2
    crit = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4)
    dl = DataLoader(BladeDS(train_df, train_tf), batch_size=32, shuffle=True, num_workers=2)
    m.train()
    for ep in range(epochs):
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = crit(m(x), y); loss.backward(); opt.step()
    return m

@torch.no_grad()
def predict(m, df):
    m.eval()
    dl = DataLoader(BladeDS(df, eval_tf), batch_size=64, num_workers=2)
    out = []
    for x, _ in dl:
        out.append(m(x.to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(out)

def evaluate(m, test_df):
    pred = predict(m, test_df)
    y = test_df.label.values
    dmg = y == 1
    recall = (pred[dmg] == 1).mean()                       # caught / all real defects
    prec = (y[pred == 1] == 1).mean() if (pred == 1).any() else 0.0
    acc = (pred == y).mean()
    per_type = {}
    for d in DEFECTS:
        sel = (test_df.defect == d).values
        if sel.any():
            per_type[d] = (pred[sel] == 1).mean()
    return {'recall': recall, 'precision': prec, 'accuracy': acc, 'per_type': per_type}

## 6. The experiment

For each amount of real damaged data `N`, train under two conditions and
measure recall on the locked real test set, averaged over several seeds:

- **synthetic + N real** — the full synthetic damage pool plus `N` real defects
- **real only (N)** — only `N` real defects as the damaged class (no synthetic)

`N = 0` for "real only" is skipped (nothing to train on). The clean/negative
class is the synthetic clean pool throughout, so the only thing changing is how
many *damaged* examples the model sees and whether they're synthetic.

In [ ]:
def build_train(n_real, use_synth, seed):
    r = np.random.default_rng(seed)
    parts = []
    if use_synth:
        parts.append(synth_damaged[['path', 'label']])
    if n_real > 0:
        take = real_pool_damaged.sample(n=min(n_real, len(real_pool_damaged)),
                                        random_state=seed)[['path', 'label']]
        parts.append(take)
    damaged = pd.concat(parts) if parts else pd.DataFrame(columns=['path', 'label'])
    # negatives: synthetic clean, capped near the number of positives for balance
    n_neg = min(len(synth_clean), max(len(damaged), 200))
    clean = synth_clean.sample(n=n_neg, random_state=seed)[['path', 'label']]
    return pd.concat([damaged, clean]).sample(frac=1, random_state=seed).reset_index(drop=True)

results = []
for seed in SEEDS:
    for n in N_REAL:
        for use_synth in [True, False]:
            if not use_synth and n == 0:
                continue
            tr = build_train(n, use_synth, seed)
            m = train_model(tr, EPOCHS, seed)
            met = evaluate(m, real_test)
            results.append({'seed': seed, 'n_real': n,
                            'condition': 'synthetic + real' if use_synth else 'real only',
                            **{k: met[k] for k in ['recall', 'precision', 'accuracy']},
                            'per_type': met['per_type']})
            print(f"seed {seed}  N={n:3d}  {results[-1]['condition']:17s}"
                  f"  recall={met['recall']:.2f}  prec={met['precision']:.2f}")
res = pd.DataFrame(results)
print('\nDone.')

## 7. Headline chart — your main slide

Recall on **real** damage as the number of real training images grows, with and
without the synthetic data. The gap between the lines at `N=0` / small `N` is
the value of the synthetic data: how much real-data collection it saves.

In [ ]:
agg = (res.groupby(['condition', 'n_real'])['recall']
          .agg(['mean', 'std']).reset_index())

plt.figure(figsize=(7, 5))
for cond, style in [('synthetic + real', 'o-'), ('real only', 's--')]:
    d = agg[agg.condition == cond].sort_values('n_real')
    plt.errorbar(d.n_real, d['mean'], yerr=d['std'].fillna(0),
                 fmt=style, capsize=3, label=cond)
plt.xlabel('Number of REAL damaged images used in training')
plt.ylabel('Recall on real damage (fraction of defects caught)')
plt.title('Does synthetic damage reduce the need for real defect photos?')
plt.ylim(0, 1); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout()
plt.savefig('headline_chart.png', dpi=150)
plt.show()
print('Saved headline_chart.png — download it for your handout.')

## 8. Which damage types did synthetic data help with?

This is often the most interesting finding: simple drawn-on damage tends to help
for geometric defects (holes, notches, missing chunks) more than for
texture-heavy ones (burns). That difference is the natural lead-in to talking
about NVIDIA Cosmos, which generates realistic *texture*.

In [ ]:
rows = []
for _, r in res.iterrows():
    for d, v in r['per_type'].items():
        rows.append({'condition': r['condition'], 'n_real': r['n_real'], 'defect': d, 'recall': v})
pt = pd.DataFrame(rows)
# compare at N=0 synthetic-only vs the best real-only run, per defect type
synth0 = pt[(pt.condition == 'synthetic + real') & (pt.n_real == 0)].groupby('defect')['recall'].mean()
print('Recall by defect type, trained on SYNTHETIC ONLY (N=0 real):')
print(synth0.round(2).to_string())

plt.figure(figsize=(7, 4))
synth0.sort_values().plot(kind='barh')
plt.xlabel('Recall on real damage (synthetic-only training)')
plt.xlim(0, 1); plt.title('Synthetic damage works better for some defect types')
plt.tight_layout(); plt.savefig('per_type.png', dpi=150); plt.show()

## 9. Grad-CAM — where is the model looking?

A sanity check and a great visual: the heatmap should light up on the actual
damage, not the background. Trains one model on synthetic+all-real and shows a
few real test blades.

In [ ]:
m = train_model(build_train(100, True, 0), EPOCHS, 0)
m.eval()

feats = {}
m.layer4.register_forward_hook(lambda mod, i, o: feats.__setitem__('a', o))

def gradcam(path):
    x = eval_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)
    x.requires_grad_(True)
    out = m(x); cls = out.argmax(1)
    m.zero_grad(); out[0, cls].backward()
    a = feats['a'].detach()[0]                    # (C,h,w)
    g = a.mean((1, 2))                            # channel weights
    cam = torch.relu((g[:, None, None] * a).sum(0)).cpu().numpy()
    cam = (cam - cam.min()) / (cam.ptp() + 1e-8)
    import cv2
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    base = np.array(Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    heat = cv2.applyColorMap((cam * 255).astype('uint8'), cv2.COLORMAP_JET)[..., ::-1]
    return (0.5 * base + 0.5 * heat).astype('uint8'), int(cls)

samples = real_test[real_test.label == 1].sample(n=4, random_state=1)['path']
fig, ax = plt.subplots(1, 4, figsize=(14, 4))
for i, p in enumerate(samples):
    img, cls = gradcam(p)
    ax[i].imshow(img); ax[i].axis('off')
    ax[i].set_title('pred: ' + ('damaged' if cls == 1 else 'good'))
plt.tight_layout(); plt.savefig('gradcam.png', dpi=150); plt.show()

## 10. Save your outputs

Download these for the handout. Right-click each in the file browser (📁 on the
left), or run the cell to copy them to Drive.

In [ ]:
import shutil
for f in ['headline_chart.png', 'per_type.png', 'gradcam.png']:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print('copied', f, '-> Drive')
res.drop(columns=['per_type']).to_csv(os.path.join(DRIVE_DIR, 'results.csv'), index=False)
print('saved results.csv to Drive')